# Two-group representational dimensionality (PR)

Group-level analysis of FLT2's per-subject participation ratio (PR) for tonecat trials: learners (non-Mandarin) vs. native listeners (Mandarin). `09_integration/TWO_GROUP_MANUSCRIPT_PLAN.md` R5.

The per-subject pipeline stays in FLT2 (GLMsingle → `mask_glmsingle_betas.py` → `dimensionality_roi.py`, fMRIPrep 25.2.5, T1w). This notebook only reads its `stage-*/sub-*_dimensionality_*.csv` outputs. `multivariate_data.py` turns them into the same long table `extract_roi_values.py` writes, so the two-group functions in `06_univariate/group_stats.py` run unchanged; PR sits in the `beta` column with `contrast = 'PR'`.

**Key question:** is the early → final PR decline learner-specific (learning-related compression) or shared (repetition/adaptation)?

**Before trusting anything here** (plan R5): FLT02/FLT22 present, every participant labelled, and the FLT2 notebook fixes done. Cortical ROIs only until FLT2 masks the striatum (`--atlas=tian-S2`).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

sys.path.insert(0, os.path.join('..', '06_univariate'))
import group_stats as gs
import multivariate_data as mvd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

bidsroot = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised'
dim_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, 'representational_dimensionality')
out_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, 'stats_two-group')
os.makedirs(out_dir, exist_ok=True)

# resampling settings, matching robustness_checks.py
n_perms, n_boot, seed = 10000, 10000, 0
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}   # as in group_level_two_group.ipynb
labels = {'non-Mandarin': 'learners', 'Mandarin': 'natives'}

## 1. Load and check

In [ ]:
participants = mvd.load_participants(bidsroot)
pr = mvd.add_composites(mvd.load_dimensionality(dim_dir, participants, 'PR'))
dsheng = mvd.add_composites(mvd.load_dimensionality(dim_dir, participants, 'Dsheng'))

print(participants.groupby('participant_group').participant_id.count().rename('in participants.tsv'))
missing = sorted(set(participants.participant_id) - set(pr.participant_id))
print('no PR output:', missing)
gs.coverage(pr)

Trials per stage (expect ~96: 16 stimuli × ~3 repetitions × 2 runs) and voxels per ROI. PR can't exceed min(16 stimuli, n voxels), so a small ROI caps it.

In [ ]:
ntrials = mvd.load_dimensionality(dim_dir, participants, 'ntrials')
print(ntrials.groupby(['participant_group', 'learning_stage']).beta.describe()[['min', 'mean', 'max']])
nvoxels = mvd.load_nvoxels(dim_dir, participants)
nvoxels.groupby('region_hemi').n_voxels.min().sort_values().head()

## 2. Stage profiles by group

Hemispheres averaged per participant; mean ± SEM.

In [ ]:
bilat = gs.bilateral_mean(pr)
regions = list(mvd.COMPOSITES) + [r for r in bilat.region.unique() if r not in mvd.COMPOSITES]
fig, axes = plt.subplots(1, len(regions), figsize=(2.4 * len(regions), 2.8), sharey=True)
for ax, region in zip(axes, regions):
    sub = bilat[bilat.region == region]
    for group in gs.GROUPS:
        g = sub[sub.participant_group == group].groupby('learning_stage').beta
        mean, sem = g.mean().reindex(gs.STAGE_ORDER), g.sem().reindex(gs.STAGE_ORDER)
        ax.errorbar(range(3), mean, yerr=sem, marker='o', capsize=3, color=colors[group], label=labels[group])
    ax.set_xticks(range(3), gs.STAGE_ORDER)
    ax.set_title(region)
axes[0].set_ylabel('participation ratio')
axes[0].legend(frameon=False)
fig.tight_layout()

## 3. Planned test: PR trend, learners vs. natives

Per-participant `[-1, 0, 1]` trend (final − early), Primary composite (HG + PT + PP, hemispheres pooled), the same battery as the striatal planned test: within each group (t, exact sign-flip, LOSO, bootstrap) and between groups (Welch, group-label permutation, LOSO, bootstrap CI). A negative trend is a decline. Association composite alongside.

The composite is the mean of the member regions' PR; confirm it matches how FLT2's notebook builds its Primary composite.

In [ ]:
trends = gs.subject_trends(pr)
planned = pd.concat([gs.trend_group_difference(trends, 'PR', region, n_perms=n_perms, n_boot=n_boot, seed=seed)
                     for region in mvd.COMPOSITES], ignore_index=True)
planned.to_csv(os.path.join(out_dir, 'pr_trend_planned.tsv'), sep='\t', index=False)
planned.round(3)

## 4. Mixed model: group × stage × level

`PR ~ group * stage * level + (1 | participant)` on the two composites (plan R5), replacing FLT2's per-stage two-way decompositions. Stage as a categorical factor (early = reference).

In [ ]:
composite = bilat[bilat.region.isin(mvd.COMPOSITES)].rename(
    columns={'beta': 'PR', 'participant_group': 'group', 'learning_stage': 'stage', 'region': 'level'})
composite['stage'] = pd.Categorical(composite.stage, categories=gs.STAGE_ORDER)
mixed = smf.mixedlm('PR ~ C(group, Treatment("Mandarin")) * stage * level',
                    composite, groups=composite.participant_id).fit(reml=True)
mixed.summary()

## 5. Exploratory: per-region tests

FDR across regions within each family. Stage-averaged group differences, trend group differences and the group × stage ANOVA.

In [ ]:
per_region = pr[~pr.region.isin(mvd.COMPOSITES)]
level_diff = gs.group_difference_tests(per_region, contrasts=('PR',), n_perms=n_perms, seed=seed)
trend_diff = gs.trend_difference_tests(gs.subject_trends(per_region), contrasts=('PR',), n_perms=n_perms, seed=seed)
anova = gs.group_by_stage_anova(per_region)
for name, table in [('pr_group-difference', level_diff), ('pr_trend-difference', trend_diff),
                    ('pr_group-by-stage-anova', anova)]:
    table.to_csv(os.path.join(out_dir, f'{name}.tsv'), sep='\t', index=False)
level_diff.round(3)

In [ ]:
trend_diff.round(3)

In [ ]:
anova[anova.q < .05].round(3)

## 6. Robustness: D_sheng

The planned test again on Sheng's dimensionality estimate (plan R5: PR primary, ROI-level D_sheng as robustness).

In [ ]:
dsheng_trends = gs.subject_trends(dsheng)
pd.concat([gs.trend_group_difference(dsheng_trends, 'Dsheng', region, n_perms=n_perms, n_boot=n_boot, seed=seed)
           for region in mvd.COMPOSITES], ignore_index=True).round(3)

## 7. Export for analysis B

Per-participant PR trends (every region and composite) for `integration_B` (plan R7).

In [ ]:
trends.to_csv(os.path.join(out_dir, 'pr_trend_scalars.tsv'), sep='\t', index=False)
print('wrote', os.path.join(out_dir, 'pr_trend_scalars.tsv'))